# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_02 — HAR Realized Volatility Forecasting

### Research question

How can realised volatility be forecast using heterogeneous time-horizon components, and how does HAR-RV compare with simple volatility forecasting baselines?

This notebook follows naturally from:

```text
01_10_realized_volatility_estimators
03_01_garch_volatility_modeling
```

The previous notebook introduced GARCH as a latent conditional-volatility model. This notebook uses observed realised-volatility measures and the **Heterogeneous Autoregressive Realized Volatility** model:

$$
\begin{aligned}
RV_{t+1} &= \beta_0 \\
&\quad + \beta_d RV_t \\
&\quad + \beta_w RV_{t,w} \\
&\quad + \beta_m RV_{t,m} \\
&\quad + \varepsilon_{t+1}
\end{aligned}
$$

It covers:

1. realised volatility and realised variance;
2. HAR-RV daily/weekly/monthly components;
3. log-HAR-RV;
4. synthetic realised-volatility data generation;
5. optional loading from the Phase 1 realised-volatility notebook;
6. train/test time-series split;
7. OLS estimation from scratch;
8. expanding-window forecasting;
9. rolling-window forecasting;
10. naïve, EWMA, and rolling-volatility baselines;
11. forecast evaluation by MSE, MAE, QLIKE, and directional accuracy;
12. volatility regime diagnostics;
13. feature importance interpretation;
14. saved forecasts, metrics, and manifest.

The key idea:

> HAR-RV is simple, interpretable, and surprisingly strong because volatility is persistent across heterogeneous horizons: daily, weekly, and monthly.

## 1. Why HAR-RV?

Volatility is persistent.

A high-volatility day often follows a high-volatility day, but volatility also has slower-moving weekly and monthly components.

The HAR-RV model represents this with three components:

$$
RV_t^{(d)} = RV_t
$$

$$
RV_t^{(w)} = \frac{1}{5}\sum_{i=0}^{4} RV_{t-i}
$$

$$
RV_t^{(m)} = \frac{1}{22}\sum_{i=0}^{21} RV_{t-i}
$$

Then:

$$
\begin{aligned}
RV_{t+1} &= \beta_0 \\
&\quad + \beta_d RV_t^{(d)} \\
&\quad + \beta_w RV_t^{(w)} \\
&\quad + \beta_m RV_t^{(m)} \\
&\quad + \varepsilon_{t+1}
\end{aligned}
$$

This is not literally an AR(22), but it approximates long-memory volatility behaviour with a low-dimensional linear regression.

## 2. Level HAR versus log HAR

Realised variance is positive and often highly skewed.

Two common modelling choices are:

### Level HAR

$$
\begin{aligned}
RV_{t+1} &= \beta_0 \\
&\quad + \beta_d RV_t \\
&\quad + \beta_w RV_{t,w} \\
&\quad + \beta_m RV_{t,m} \\
&\quad + \varepsilon_{t+1}
\end{aligned}
$$

### Log HAR

$$
\begin{aligned}
\log RV_{t+1} &= \beta_0 \\
&\quad + \beta_d \log RV_t \\
&\quad + \beta_w \log RV_{t,w} \\
&\quad + \beta_m \log RV_{t,m} \\
&\quad + \varepsilon_{t+1}
\end{aligned}
$$

The log model often behaves better because it respects positivity after exponentiation and compresses extreme values.

However, back-transforming log forecasts introduces bias:

$$
\mathbb E[RV]\neq \exp(\mathbb E[\log RV])
$$

This notebook compares both approaches transparently.

## 3. Forecast evaluation

Volatility forecast evaluation is not only ordinary MSE.

We use:

### MSE

$$
\frac{1}{N}\sum_t(\hat{RV}_t-RV_t)^2
$$

### MAE

$$
\frac{1}{N}\sum_t|\hat{RV}_t-RV_t|
$$

### QLIKE

$$
\begin{aligned}
QLIKE_t &= \frac{RV_t}{\hat{RV}_t} \\
&\quad - \log \Big( \frac{RV_t}{\hat{RV}_t} \Big) \\
&\quad - 1
\end{aligned}
$$

QLIKE is popular for volatility forecast evaluation because it is robust to noisy volatility proxies and penalises underprediction strongly.

We also report directional accuracy for volatility changes:

$$
\operatorname{sign}(\hat{RV}_{t+1}-RV_t) = \operatorname{sign}(RV_{t+1}-RV_t)
$$

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class HARRVConfig:
    n_obs: int = 2_500
    train_fraction: float = 0.70
    weekly_window: int = 5
    monthly_window: int = 22
    ewma_lambda: float = 0.94
    rolling_window: int = 22
    expanding_min_train: int = 500
    rolling_train_window: int = 750
    seed: int = 42


config = HARRVConfig()
config

## 5. Data source

This notebook first tries to load a HAR-ready realised-volatility file from Phase 1:

```text
data/processed/realized_volatility_estimators_v1/har_rv_ready_rogers_satchell.csv
```

If that file is not available, it generates a synthetic realised-volatility dataset with:

- persistent latent log volatility;
- noisy realised variance proxy;
- volatility spikes;
- heavy-tailed return shocks.

This keeps the notebook runnable as a standalone file while still connecting to the repo pipeline.

In [ ]:
def simulate_realized_volatility_dataset(config: HARRVConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs

    dates = pd.bdate_range("2015-01-01", periods=n)

    # Persistent latent log variance process.
    log_var = np.zeros(n)
    long_run_log_var = np.log((0.18 / np.sqrt(252)) ** 2)
    phi = 0.985
    vol_of_log_var = 0.13

    log_var[0] = long_run_log_var

    for t in range(1, n):
        log_var[t] = (
            (1 - phi) * long_run_log_var
            + phi * log_var[t - 1]
            + vol_of_log_var * rng.standard_normal()
        )

        # Occasional volatility shock.
        if rng.uniform() < 0.012:
            log_var[t] += rng.uniform(0.5, 1.2)

    latent_variance = np.exp(log_var)
    latent_vol = np.sqrt(latent_variance)

    # Heavy-tailed returns.
    z = rng.standard_t(df=7, size=n) * np.sqrt((7 - 2) / 7)
    returns = latent_vol * z

    # Noisy realised variance proxy.
    measurement_noise = rng.lognormal(mean=-0.5 * 0.35**2, sigma=0.35, size=n)
    realized_variance = latent_variance * measurement_noise

    out = pd.DataFrame({
        "timestamp": dates,
        "return": returns,
        "latent_variance": latent_variance,
        "latent_annualized_vol": latent_vol * np.sqrt(252),
        "rv_daily": realized_variance
    })

    out["price"] = 100 * np.exp(out["return"].cumsum())

    return out


def load_or_simulate_har_data(config: HARRVConfig) -> tuple[pd.DataFrame, str]:
    path = Path("data/processed/realized_volatility_estimators_v1/har_rv_ready_rogers_satchell.csv")

    if path.exists():
        raw = pd.read_csv(path)

        # Normalise expected columns.
        if "timestamp" not in raw.columns:
            raw["timestamp"] = pd.bdate_range("2015-01-01", periods=len(raw))

        if "rv_daily" not in raw.columns:
            candidate_cols = [c for c in raw.columns if c.startswith("rv_") and "daily" in c]
            if candidate_cols:
                raw["rv_daily"] = raw[candidate_cols[0]]
            elif "rv_next_day_target" in raw.columns:
                raw["rv_daily"] = raw["rv_next_day_target"].shift(1)
            else:
                raise ValueError("Could not identify realised variance column.")

        raw["timestamp"] = pd.to_datetime(raw["timestamp"])
        raw = raw.dropna(subset=["rv_daily"]).reset_index(drop=True)
        return raw, "loaded_phase1_har_ready_rogers_satchell"

    return simulate_realized_volatility_dataset(config), "synthetic_fallback"


rv_raw, data_source = load_or_simulate_har_data(config)

data_source, rv_raw.head()

## 6. Visualise realised volatility

We plot:

1. price;
2. daily returns if available;
3. realised annualised volatility.

Annualised realised volatility is:

$$
\sqrt{252\cdot RV_t}
$$

In [ ]:
rv_data = rv_raw.copy()
rv_data["rv_daily"] = rv_data["rv_daily"].clip(lower=1e-12)
rv_data["rv_annualized_vol"] = np.sqrt(252 * rv_data["rv_daily"])

if "timestamp" not in rv_data:
    rv_data["timestamp"] = pd.bdate_range("2015-01-01", periods=len(rv_data))

if "return" not in rv_data:
    rv_data["return"] = np.nan

if "price" not in rv_data and rv_data["return"].notna().all():
    rv_data["price"] = 100 * np.exp(rv_data["return"].cumsum())

rv_data.head()

In [ ]:
if "price" in rv_data:
    plt.figure(figsize=(12, 6))
    plt.plot(rv_data["timestamp"], rv_data["price"])
    plt.title("Price Series")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.show()

plt.figure(figsize=(12, 6))
plt.plot(rv_data["timestamp"], rv_data["rv_annualized_vol"])
plt.title("Realised Annualised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.show()

## 7. HAR feature construction

We create:

$$
RV_d = RV_t
$$

$$
RV_w = \frac{1}{5}\sum_{i=0}^{4}RV_{t-i}
$$

$$
RV_m = \frac{1}{22}\sum_{i=0}^{21}RV_{t-i}
$$

The one-step-ahead target is:

$$
RV_{t+1}
$$

We also create log versions:

$$
\log RV_d,\quad \log RV_w,\quad \log RV_m,\quad \log RV_{t+1}
$$

In [ ]:
def add_har_features(df: pd.DataFrame, config: HARRVConfig) -> pd.DataFrame:
    out = df.sort_values("timestamp").copy()

    out["rv_daily"] = out["rv_daily"].clip(lower=1e-12)
    out["rv_weekly"] = out["rv_daily"].rolling(config.weekly_window).mean()
    out["rv_monthly"] = out["rv_daily"].rolling(config.monthly_window).mean()

    out["rv_next"] = out["rv_daily"].shift(-1)

    for col in ["rv_daily", "rv_weekly", "rv_monthly", "rv_next"]:
        out[f"log_{col}"] = np.log(out[col].clip(lower=1e-12))

    out["rv_change_direction"] = np.sign(out["rv_next"] - out["rv_daily"])

    feature_cols = [
        "rv_daily",
        "rv_weekly",
        "rv_monthly",
        "log_rv_daily",
        "log_rv_weekly",
        "log_rv_monthly",
        "rv_next",
        "log_rv_next"
    ]

    out = out.dropna(subset=feature_cols).reset_index(drop=True)

    return out


har_df = add_har_features(rv_data, config)

har_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(har_df["timestamp"], np.sqrt(252 * har_df["rv_daily"]), label="Daily RV vol", alpha=0.6)
plt.plot(har_df["timestamp"], np.sqrt(252 * har_df["rv_weekly"]), label="Weekly RV vol", alpha=0.8)
plt.plot(har_df["timestamp"], np.sqrt(252 * har_df["rv_monthly"]), label="Monthly RV vol", linewidth=2)
plt.title("HAR Components as Annualised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 8. Train/test split

This is a time-series problem.

We do not shuffle.

Train set:

$$
[0, T_{\text{split}}]
$$

Test set:

$$
(T_{\text{split}}, T]
$$

In [ ]:
split_idx = int(len(har_df) * config.train_fraction)

train_df = har_df.iloc[:split_idx].copy()
test_df = har_df.iloc[split_idx:].copy()

pd.Series({
    "n_total": len(har_df),
    "n_train": len(train_df),
    "n_test": len(test_df),
    "train_start": train_df["timestamp"].iloc[0],
    "train_end": train_df["timestamp"].iloc[-1],
    "test_start": test_df["timestamp"].iloc[0],
    "test_end": test_df["timestamp"].iloc[-1]
})

## 9. OLS estimation from scratch

We implement OLS directly:

$$
\hat\beta=(X^\top X)^{-1}X^\top y
$$

with a small ridge fallback for numerical stability.

This keeps the notebook dependency-light and makes the regression transparent.

In [ ]:
def fit_ols(X: np.ndarray, y: np.ndarray, ridge: float = 1e-10) -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    xtx = X.T @ X
    xty = X.T @ y

    try:
        beta = np.linalg.solve(xtx, xty)
    except np.linalg.LinAlgError:
        beta = np.linalg.solve(xtx + ridge * np.eye(X.shape[1]), xty)

    fitted = X @ beta
    residuals = y - fitted

    n, k = X.shape
    sigma2 = float((residuals @ residuals) / max(n - k, 1))
    cov_beta = sigma2 * np.linalg.pinv(xtx)
    se_beta = np.sqrt(np.diag(cov_beta))

    return {
        "beta": beta,
        "se_beta": se_beta,
        "fitted": fitted,
        "residuals": residuals,
        "sigma2": sigma2,
        "n": n,
        "k": k
    }


def make_design(df: pd.DataFrame, cols: list[str]) -> np.ndarray:
    return np.column_stack([np.ones(len(df)), df[cols].to_numpy()])


level_feature_cols = ["rv_daily", "rv_weekly", "rv_monthly"]
log_feature_cols = ["log_rv_daily", "log_rv_weekly", "log_rv_monthly"]

X_train_level = make_design(train_df, level_feature_cols)
y_train_level = train_df["rv_next"].to_numpy()

X_train_log = make_design(train_df, log_feature_cols)
y_train_log = train_df["log_rv_next"].to_numpy()

level_fit = fit_ols(X_train_level, y_train_level)
log_fit = fit_ols(X_train_log, y_train_log)

pd.DataFrame({
    "term": ["intercept"] + level_feature_cols,
    "level_beta": level_fit["beta"],
    "level_se": level_fit["se_beta"]
})

In [ ]:
pd.DataFrame({
    "term": ["intercept"] + log_feature_cols,
    "log_beta": log_fit["beta"],
    "log_se": log_fit["se_beta"]
})

## 10. Static train/test forecasts

We first fit once on the training set and forecast the test set.

Level HAR forecast:

$$
\widehat{RV}_{t+1}=X_t\hat\beta
$$

Log HAR forecast:

$$
\widehat{\log RV}_{t+1}=X_t\hat\beta
$$

and:

$$
\widehat{RV}_{t+1}=\exp(\widehat{\log RV}_{t+1})
$$

In [ ]:
def predict_ols(fit: dict, X: np.ndarray) -> np.ndarray:
    return X @ fit["beta"]


X_test_level = make_design(test_df, level_feature_cols)
X_test_log = make_design(test_df, log_feature_cols)

test_forecasts = test_df[["timestamp", "rv_daily", "rv_next", "log_rv_next"]].copy()
test_forecasts["forecast_level_har"] = np.maximum(predict_ols(level_fit, X_test_level), 1e-12)
test_forecasts["forecast_log_har"] = np.exp(predict_ols(log_fit, X_test_log))

test_forecasts.head()

## 11. Forecast metrics

We use:

$$
MSE,\quad MAE,\quad QLIKE
$$

and directional accuracy.

Direction is defined as forecasting whether volatility rises or falls relative to today's realised variance.

In [ ]:
def qlike(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).clip(min=1e-12)
    y_pred = np.asarray(y_pred, dtype=float).clip(min=1e-12)

    ratio = y_true / y_pred
    return ratio - np.log(ratio) - 1.0


def forecast_metrics(df: pd.DataFrame, forecast_col: str, target_col: str = "rv_next") -> dict:
    y = df[target_col].to_numpy().clip(min=1e-12)
    pred = df[forecast_col].to_numpy().clip(min=1e-12)

    mse = float(np.mean((pred - y) ** 2))
    mae = float(np.mean(np.abs(pred - y)))
    ql = float(np.mean(qlike(y, pred)))

    actual_direction = np.sign(y - df["rv_daily"].to_numpy())
    predicted_direction = np.sign(pred - df["rv_daily"].to_numpy())
    directional_accuracy = float(np.mean(actual_direction == predicted_direction))

    return {
        "forecast": forecast_col,
        "mse": mse,
        "mae": mae,
        "qlike": ql,
        "directional_accuracy": directional_accuracy,
        "mean_forecast_ann_vol": float(np.mean(np.sqrt(252 * pred))),
        "mean_realized_ann_vol": float(np.mean(np.sqrt(252 * y)))
    }


static_metrics = pd.DataFrame([
    forecast_metrics(test_forecasts, "forecast_level_har"),
    forecast_metrics(test_forecasts, "forecast_log_har")
])

static_metrics

## 12. Baseline forecasts

We compare HAR models against simple baselines.

### Naïve forecast

$$
\widehat{RV}_{t+1}=RV_t
$$

### Weekly average

$$
\widehat{RV}_{t+1}=RV_{t,w}
$$

### Monthly average

$$
\widehat{RV}_{t+1}=RV_{t,m}
$$

### EWMA

$$
\hat h_t=\lambda \hat h_{t-1}+(1-\lambda)RV_t
$$

In [ ]:
def add_baseline_forecasts(df: pd.DataFrame, config: HARRVConfig) -> pd.DataFrame:
    out = df.copy()

    out["forecast_naive"] = out["rv_daily"]
    out["forecast_weekly"] = out["rv_weekly"]
    out["forecast_monthly"] = out["rv_monthly"]

    ewma = np.zeros(len(out))
    ewma[0] = out["rv_daily"].iloc[0]

    for i in range(1, len(out)):
        ewma[i] = config.ewma_lambda * ewma[i - 1] + (1 - config.ewma_lambda) * out["rv_daily"].iloc[i - 1]

    out["forecast_ewma"] = ewma

    out["forecast_rolling_22"] = out["rv_daily"].rolling(config.rolling_window).mean()
    out["forecast_rolling_22"] = out["forecast_rolling_22"].fillna(out["rv_monthly"])

    return out


test_forecasts = add_baseline_forecasts(test_forecasts.merge(
    test_df[["timestamp", "rv_weekly", "rv_monthly"]],
    on="timestamp",
    how="left"
), config)

baseline_cols = [
    "forecast_naive",
    "forecast_weekly",
    "forecast_monthly",
    "forecast_ewma",
    "forecast_rolling_22"
]

baseline_metrics = pd.DataFrame([
    forecast_metrics(test_forecasts, col)
    for col in baseline_cols
])

all_static_metrics = pd.concat([static_metrics, baseline_metrics], ignore_index=True).sort_values("qlike")

all_static_metrics

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test_forecasts["timestamp"], np.sqrt(252 * test_forecasts["rv_next"]), label="Realised next vol", alpha=0.65)
plt.plot(test_forecasts["timestamp"], np.sqrt(252 * test_forecasts["forecast_log_har"]), label="Log HAR forecast", alpha=0.85)
plt.plot(test_forecasts["timestamp"], np.sqrt(252 * test_forecasts["forecast_ewma"]), label="EWMA forecast", alpha=0.75)
plt.title("Volatility Forecasts vs Realised Next Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 13. Expanding-window forecasts

A static train/test fit is simple but not how a live system usually works.

An expanding-window forecast refits using all data available up to time $t$:

$$
\{1,\dots,t\}
$$

and predicts:

$$
RV_{t+1}
$$

This is closer to a live research pipeline.

In [ ]:
def expanding_window_har_forecasts(df: pd.DataFrame, config: HARRVConfig) -> pd.DataFrame:
    rows = []

    for i in range(config.expanding_min_train, len(df)):
        train = df.iloc[:i].copy()
        current = df.iloc[[i]].copy()

        X_train_log = make_design(train, log_feature_cols)
        y_train_log = train["log_rv_next"].to_numpy()

        fit = fit_ols(X_train_log, y_train_log)

        X_current = make_design(current, log_feature_cols)
        pred_log = predict_ols(fit, X_current)[0]
        pred_rv = float(np.exp(pred_log))

        rows.append({
            "timestamp": current["timestamp"].iloc[0],
            "rv_daily": current["rv_daily"].iloc[0],
            "rv_next": current["rv_next"].iloc[0],
            "forecast_expanding_log_har": pred_rv
        })

    return pd.DataFrame(rows)


expanding_forecasts = expanding_window_har_forecasts(har_df, config)

expanding_forecasts.head()

In [ ]:
expanding_metrics = pd.DataFrame([
    forecast_metrics(
        expanding_forecasts,
        "forecast_expanding_log_har"
    )
])

expanding_metrics

## 14. Rolling-window forecasts

A rolling-window forecast uses a fixed-length recent window:

$$
\{t-W+1,\dots,t\}
$$

This can adapt faster if parameters change over time.

It may be less stable than expanding-window estimation.

In [ ]:
def rolling_window_har_forecasts(df: pd.DataFrame, config: HARRVConfig) -> pd.DataFrame:
    rows = []
    W = config.rolling_train_window

    for i in range(W, len(df)):
        train = df.iloc[i - W:i].copy()
        current = df.iloc[[i]].copy()

        X_train_log = make_design(train, log_feature_cols)
        y_train_log = train["log_rv_next"].to_numpy()

        fit = fit_ols(X_train_log, y_train_log)

        X_current = make_design(current, log_feature_cols)
        pred_log = predict_ols(fit, X_current)[0]
        pred_rv = float(np.exp(pred_log))

        rows.append({
            "timestamp": current["timestamp"].iloc[0],
            "rv_daily": current["rv_daily"].iloc[0],
            "rv_next": current["rv_next"].iloc[0],
            "forecast_rolling_log_har": pred_rv
        })

    return pd.DataFrame(rows)


rolling_forecasts = rolling_window_har_forecasts(har_df, config)

rolling_forecasts.head()

In [ ]:
rolling_metrics = pd.DataFrame([
    forecast_metrics(
        rolling_forecasts,
        "forecast_rolling_log_har"
    )
])

rolling_metrics

## 15. Combined forecast comparison

We compare:

1. static HAR;
2. static log-HAR;
3. baselines;
4. expanding log-HAR;
5. rolling log-HAR.

Because expanding and rolling forecasts may start at different dates, metrics are not perfectly identical sample comparisons. In a production backtest, align forecast dates exactly.

In [ ]:
combined_metrics = pd.concat([
    all_static_metrics,
    expanding_metrics,
    rolling_metrics
], ignore_index=True).sort_values("qlike")

combined_metrics

In [ ]:
plt.figure(figsize=(10, 6))
plot_metrics = combined_metrics.sort_values("qlike")
plt.barh(plot_metrics["forecast"], plot_metrics["qlike"])
plt.title("Forecast Comparison by QLIKE")
plt.xlabel("QLIKE, lower is better")
plt.ylabel("Forecast")
plt.show()

plt.figure(figsize=(10, 6))
plot_metrics = combined_metrics.sort_values("mae")
plt.barh(plot_metrics["forecast"], plot_metrics["mae"])
plt.title("Forecast Comparison by MAE")
plt.xlabel("MAE, lower is better")
plt.ylabel("Forecast")
plt.show()

## 16. Residual diagnostics

Forecast residuals:

$$
e_{t+1}=RV_{t+1}-\widehat{RV}_{t+1}
$$

For log HAR:

$$
e_{t+1}^{log}=\log RV_{t+1}-\widehat{\log RV}_{t+1}
$$

We inspect whether forecast errors are biased and whether errors are larger in high-volatility regimes.

In [ ]:
test_forecasts["error_log_har"] = test_forecasts["rv_next"] - test_forecasts["forecast_log_har"]
test_forecasts["abs_error_log_har"] = test_forecasts["error_log_har"].abs()
test_forecasts["realized_next_ann_vol"] = np.sqrt(252 * test_forecasts["rv_next"])
test_forecasts["forecast_log_har_ann_vol"] = np.sqrt(252 * test_forecasts["forecast_log_har"])

residual_summary = pd.Series({
    "mean_error": test_forecasts["error_log_har"].mean(),
    "median_error": test_forecasts["error_log_har"].median(),
    "mean_abs_error": test_forecasts["abs_error_log_har"].mean(),
    "corr_abs_error_realized_vol": test_forecasts["abs_error_log_har"].corr(test_forecasts["realized_next_ann_vol"])
})

residual_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(test_forecasts["forecast_log_har_ann_vol"], test_forecasts["realized_next_ann_vol"], alpha=0.35)
min_v = min(test_forecasts["forecast_log_har_ann_vol"].min(), test_forecasts["realized_next_ann_vol"].min())
max_v = max(test_forecasts["forecast_log_har_ann_vol"].max(), test_forecasts["realized_next_ann_vol"].max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Log-HAR Forecast Vol vs Realised Next Vol")
plt.xlabel("Forecast annualised vol")
plt.ylabel("Realised next annualised vol")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(test_forecasts["error_log_har"], bins=80)
plt.axvline(0, linestyle="--")
plt.title("Log-HAR Forecast Error Distribution")
plt.xlabel("RV forecast error")
plt.ylabel("Count")
plt.show()

## 17. Regime diagnostics

HAR-RV can behave differently in high- and low-volatility regimes.

We split the test set by realised volatility quartiles and compute errors by regime.

In [ ]:
test_forecasts["realized_vol_quartile"] = pd.qcut(
    test_forecasts["realized_next_ann_vol"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"]
)

regime_error = (
    test_forecasts
    .groupby("realized_vol_quartile", observed=True)
    .agg(
        n=("rv_next", "count"),
        mean_realized_ann_vol=("realized_next_ann_vol", "mean"),
        mean_forecast_ann_vol=("forecast_log_har_ann_vol", "mean"),
        mean_abs_error=("abs_error_log_har", "mean"),
        qlike=("rv_next", lambda x: np.nan)
    )
    .reset_index()
)

# Compute QLIKE manually per regime.
qlike_values = []
for regime, group in test_forecasts.groupby("realized_vol_quartile", observed=True):
    ql = float(np.mean(qlike(group["rv_next"], group["forecast_log_har"])))
    qlike_values.append((regime, ql))

qlike_map = dict(qlike_values)
regime_error["qlike"] = regime_error["realized_vol_quartile"].map(qlike_map).astype(float)

regime_error

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(regime_error["realized_vol_quartile"].astype(str), regime_error["qlike"])
plt.title("Log-HAR QLIKE by Realised Volatility Regime")
plt.xlabel("Realised volatility regime")
plt.ylabel("QLIKE")
plt.show()

## 18. Feature interpretation

In HAR-RV, coefficients roughly represent how much the forecast uses:

- yesterday's volatility;
- last week's volatility;
- last month's volatility.

Positive weekly/monthly coefficients suggest persistence across horizons.

The log-HAR coefficients are often easier to interpret because they describe persistence in log volatility.

In [ ]:
coef_tables = []

coef_tables.append(pd.DataFrame({
    "model": "level_har_static",
    "term": ["intercept"] + level_feature_cols,
    "coefficient": level_fit["beta"],
    "standard_error": level_fit["se_beta"]
}))

coef_tables.append(pd.DataFrame({
    "model": "log_har_static",
    "term": ["intercept"] + log_feature_cols,
    "coefficient": log_fit["beta"],
    "standard_error": log_fit["se_beta"]
}))

coef_table = pd.concat(coef_tables, ignore_index=True)

coef_table

In [ ]:
plt.figure(figsize=(10, 6))
plot_coef = coef_table[(coef_table["model"] == "log_har_static") & (coef_table["term"] != "intercept")]
plt.bar(plot_coef["term"], plot_coef["coefficient"])
plt.title("Static Log-HAR Coefficients")
plt.xlabel("Feature")
plt.ylabel("Coefficient")
plt.show()

## 19. Toy volatility risk-scaling use case

A volatility forecast can be used to scale exposure:

$$
w_t = \frac{\sigma_{\text{target}}}{\hat\sigma_{t+1}}
$$

with a leverage cap.

This is not an alpha claim; it is a risk-control demonstration.

If volatility forecasts are useful, risk-scaled returns should have more stable realised volatility.

In [ ]:
def volatility_targeting_from_forecasts(forecast_df, forecast_col, target_vol=0.15, max_leverage=2.0):
    out = forecast_df.copy()

    forecast_ann_vol = np.sqrt(252 * out[forecast_col].clip(lower=1e-12))
    weight = target_vol / forecast_ann_vol
    weight = weight.clip(lower=0.0, upper=max_leverage)

    # Synthetic returns may be available only if data source includes return.
    # Merge returns from har_df by timestamp.
    returns_map = har_df.set_index("timestamp")["return"].to_dict()
    out["next_return"] = out["timestamp"].map(returns_map).shift(-1)
    out["weight"] = weight
    out["strategy_return"] = out["weight"] * out["next_return"]

    out = out.dropna(subset=["strategy_return"]).reset_index(drop=True)
    out["cum_strategy"] = (1 + out["strategy_return"]).cumprod()
    out["cum_underlying"] = (1 + out["next_return"]).cumprod()

    return out


if har_df["return"].notna().all():
    vt_df = volatility_targeting_from_forecasts(test_forecasts, "forecast_log_har", target_vol=0.15, max_leverage=2.0)

    vt_summary = pd.Series({
        "strategy_ann_vol": vt_df["strategy_return"].std() * np.sqrt(252),
        "underlying_ann_vol": vt_df["next_return"].std() * np.sqrt(252),
        "strategy_mean_daily": vt_df["strategy_return"].mean(),
        "underlying_mean_daily": vt_df["next_return"].mean(),
        "mean_weight": vt_df["weight"].mean(),
        "max_weight": vt_df["weight"].max()
    })
else:
    vt_df = pd.DataFrame()
    vt_summary = pd.Series(dtype=float)

vt_summary

In [ ]:
if not vt_df.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(vt_df["cum_strategy"], label="Vol-targeted")
    plt.plot(vt_df["cum_underlying"], label="Underlying")
    plt.title("Toy Volatility Targeting Using Log-HAR Forecasts")
    plt.xlabel("Time")
    plt.ylabel("Cumulative growth")
    plt.legend()
    plt.show()
else:
    print("Return series unavailable; volatility-targeting demonstration skipped.")

## 20. Practical checklist for HAR-RV forecasting

Before trusting a HAR-RV model, check:

1. **Realised volatility estimator**  
   Which RV estimator is used? Close-to-close, Parkinson, Garman-Klass, Rogers-Satchell, intraday RV?

2. **Target definition**  
   Are you forecasting variance, volatility, or log variance?

3. **Leakage**  
   Are weekly/monthly averages computed using only past data?

4. **Train/test split**  
   Is evaluation time-ordered?

5. **Model form**  
   Level HAR or log HAR?

6. **Baselines**  
   Does HAR beat naïve, EWMA, and rolling average baselines?

7. **Loss function**  
   MSE, MAE, QLIKE, or risk-specific loss?

8. **Regime behaviour**  
   Does the model underpredict high-volatility periods?

9. **Forecast use case**  
   Risk sizing, VaR, option hedging, or alpha features?

10. **Retraining protocol**  
   Static, expanding, or rolling?

## 21. Saving outputs

The notebook saves:

1. HAR-ready feature table;
2. train/test split metadata;
3. static coefficient table;
4. static forecast table;
5. baseline metrics;
6. expanding forecast table;
7. rolling forecast table;
8. combined metrics;
9. residual diagnostics;
10. regime diagnostics;
11. volatility-targeting toy output;
12. manifest.

In [ ]:
output_dir = Path("data/processed/har_realized_volatility_forecasting_v1")
output_dir.mkdir(parents=True, exist_ok=True)

har_features_path = output_dir / "har_features.csv"
split_metadata_path = output_dir / "train_test_split_metadata.json"
coef_path = output_dir / "har_coefficients.csv"
static_forecasts_path = output_dir / "static_test_forecasts.csv"
combined_metrics_path = output_dir / "forecast_metrics.csv"
expanding_path = output_dir / "expanding_log_har_forecasts.csv"
rolling_path = output_dir / "rolling_log_har_forecasts.csv"
residual_summary_path = output_dir / "residual_summary.csv"
regime_path = output_dir / "regime_error_diagnostics.csv"
vt_path = output_dir / "volatility_targeting_toy.csv"
manifest_path = output_dir / "manifest.json"

har_df.to_csv(har_features_path, index=False)
coef_table.to_csv(coef_path, index=False)
test_forecasts.to_csv(static_forecasts_path, index=False)
combined_metrics.to_csv(combined_metrics_path, index=False)
expanding_forecasts.to_csv(expanding_path, index=False)
rolling_forecasts.to_csv(rolling_path, index=False)
residual_summary.to_frame("value").to_csv(residual_summary_path)
regime_error.to_csv(regime_path, index=False)
vt_df.to_csv(vt_path, index=False)

split_metadata = {
    "n_total": int(len(har_df)),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
    "train_start": str(train_df["timestamp"].iloc[0]),
    "train_end": str(train_df["timestamp"].iloc[-1]),
    "test_start": str(test_df["timestamp"].iloc[0]),
    "test_end": str(test_df["timestamp"].iloc[-1]),
    "data_source": data_source
}

split_metadata_path.write_text(json.dumps(split_metadata, indent=2, default=str))

manifest = {
    "dataset_name": "har_realized_volatility_forecasting_outputs",
    "schema_version": "har_realized_volatility_forecasting_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "data_source": data_source,
    "split_metadata": split_metadata,
    "best_model_by_qlike": combined_metrics.sort_values("qlike").iloc[0].to_dict(),
    "residual_summary": residual_summary.to_dict(),
    "volatility_targeting_summary": vt_summary.to_dict(),
    "notes": [
        "Notebook loads Phase 1 HAR-ready realised-volatility data if available, otherwise uses synthetic fallback.",
        "HAR features use daily, weekly, and monthly realised variance components.",
        "Static, expanding, and rolling log-HAR forecasts are compared.",
        "Forecast metrics include MSE, MAE, QLIKE, and directional accuracy.",
        "Volatility-targeting section is a toy risk-scaling demonstration, not an alpha claim."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

har_features_path, combined_metrics_path, expanding_path, rolling_path, manifest_path

## 22. Limitations

### 22.1 Synthetic fallback data

If Phase 1 realised-volatility files are unavailable, this notebook uses synthetic data.

Real results depend heavily on the quality of the realised-volatility estimator.

### 22.2 Linear model

HAR-RV is linear and simple.

It may miss nonlinear regime effects, jumps, leverage effects, and macro/event features.

### 22.3 No intraday microstructure treatment

The notebook does not estimate intraday realised variance from ticks.

It assumes a realised-variance series already exists.

### 22.4 Log back-transform bias

The log-HAR forecast uses:

$$
\exp(\widehat{\log RV})
$$

without bias correction. A production model may adjust for residual variance.

### 22.5 Forecast horizons

The notebook focuses on one-step-ahead forecasts.

Multi-step forecasting requires iterative or direct horizon-specific models.

### 22.6 Evaluation alignment

Static, expanding, and rolling forecasts may have different start dates.

For formal model comparisons, align forecast timestamps exactly.

### 22.7 Vol targeting is only illustrative

Volatility targeting is risk scaling, not standalone alpha.

## 23. Research frontier and extensions

Important extensions include:

1. **HAR-RV-CJ**  
   Separate continuous and jump variation components.

2. **HARQ**  
   Add realised quarticity to account for measurement error.

3. **Leverage HAR**  
   Add negative-return indicators or signed returns.

4. **Realised GARCH**  
   Jointly model returns and realised volatility.

5. **Machine learning volatility forecasts**  
   Random forests, gradient boosting, neural nets, transformers.

6. **Option-implied volatility hybrids**  
   Combine HAR-RV with implied volatility and volatility risk premium.

7. **Multi-asset HAR**  
   Forecast volatility spillovers across assets.

8. **Chinese futures HAR-RV**  
   Handle night sessions, contract rolls, price limits, and product seasonality.

9. **Forecast combination**  
   Ensemble HAR, GARCH, implied volatility, and ML forecasts.

10. **Economic value tests**  
   Evaluate forecasts through VaR, volatility targeting, option hedging, or risk parity.

## 24. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_03_regime_detection_hidden_markov_models**  
   Detecting volatility regimes.

2. **03_04_factor_momentum_and_volatility_scaling**  
   Using volatility forecasts for risk-scaled alpha.

3. **03_05_cross_sectional_alpha_features**  
   Adding volatility features to cross-sectional models.

4. **03_06_tree_models_for_return_prediction**  
   Nonlinear ML forecasting.

5. **04_06_var_cvar_and_stress_testing**  
   Using HAR forecasts in risk models.

6. **05_01_vectorized_backtest_engine**  
   Testing forecast-driven strategies.

7. **07_01_multi_asset_cta_research_pipeline**  
   Applying HAR-RV to futures trend systems.

## 25. Summary

This notebook implemented HAR-RV forecasting.

It showed how to:

1. load or simulate realised volatility data;
2. construct daily, weekly, and monthly HAR components;
3. fit level HAR and log HAR regressions;
4. build static train/test forecasts;
5. compare against naïve, weekly, monthly, EWMA, and rolling baselines;
6. implement expanding-window forecasts;
7. implement rolling-window forecasts;
8. evaluate forecasts using MSE, MAE, QLIKE, and direction;
9. diagnose forecast errors by volatility regime;
10. interpret HAR coefficients;
11. demonstrate a toy volatility-targeting use case;
12. save forecasts, metrics, and manifests.

The key computational takeaway:

> HAR-RV is a small linear model with large practical value because its features encode volatility persistence across multiple horizons.

The key financial takeaway:

> Realised volatility forecasts are not just statistical outputs; they become inputs for risk sizing, VaR, hedging, and alpha research.

## 26. Further reading

- Corsi, F. "A Simple Approximate Long-Memory Model of Realized Volatility."
- Andersen, Bollerslev, Diebold, and Labys on realised volatility.
- Barndorff-Nielsen and Shephard on realised variation.
- Realised GARCH literature.
- HAR-RV-CJ and HARQ extensions.
- Literature on QLIKE volatility forecast evaluation.